In [1]:
import numpy as np
import pandas as pd


# 1. Asset assumptions

assets = ["Bonds", "Market ETF", "Large Cap", "Mid Cap",
          "Small Cap","Foreign Ex US", "Emerging", "Commodities"]






In [4]:
#Last 10 yr returns for each asset class collected from a dataset from Stern New york 
#https://www.stern.nyu.edu/~adamodar/pc/datasets/histretSP.xls

data = {
    "Year": [2015,2016,2017,2018,2019,2020,2021,2022,2023,2024],
    "Bonds": [1.10, 2.26, 3.54, 0.01, 8.72, 7.51, -3.54, -15.19, 4.50, 3.90],
    "Market ETF": [1.38, 11.96, 21.69, -4.38, 31.29, 18.40, 28.71, -18.14, 26.69, 24.88],
    "Large Cap": [1.38, 11.77, 21.61, -4.23, 31.21, 18.40, 28.71, -18.14, 26.69, 24.88],
    "Mid Cap": [-1.82, 17.53, 21.75, -11.15, 28.17, 15.43, 27.83, -20.76, 14.69, 17.00],
    "Small Cap": [-0.17, 22.07, 22.32, -18.68, 25.95, 19.63, 14.80, -29.41, 15.86, 21.00],
    "Foreign Ex US": [-0.81, 1.00, 25.03, -13.79, 21.47, 7.82, 11.29, -14.88, 17.78, 15.00],
    "Emerging Markets": [-14.92, 11.19, 37.75, -14.58, 18.42, 18.31, -2.54, -20.09, 4.43, 10.00],
    "Commodities": [1.06, 8.56, 13.12, -1.58, 18.30, 24.40, -3.86, -10.38, 12.58, 8.00]
}

df = pd.DataFrame(data)


# 2. Convert % to decimals

returns = df.drop(columns=["Year"]) / 100.0


# -------------------------
# 2. Covariance & Mean Returns
# -------------------------
mean_returns = returns.mean().values
std_devs = returns.std().values
cov_matrix = np.outer(std_devs, std_devs) * returns.corr().values
risk_free_rate = 0.02

print(mean_returns)







[0.01281 0.14248 0.14228 0.10867 0.09337 0.06991 0.04797 0.0702 ]


In [5]:

# -------------------------
# 3. Volatility Bands (Realistic)
# -------------------------
bands = {
    "Conservative Investors": (0.00, 0.10),
    "Balanced Investors": (0.07, 0.12),
    "Aggressive Investors": (0.12, 0.20),
    "Pre-Retirees": (0.00, 0.10),
    "Second Chance Retirees": (0.07, 0.13)
}

# -------------------------
# 4. Segment-specific Dirichlet Alphas
# -------------------------
segment_alphas = {
    "Conservative Investors": np.array([4, 2.5, 1.0, 0.6, 0.6, 0.4, 0.4, 2.0]),
    "Balanced Investors": np.array([1.5, 2.0, 2.0, 1.5, 1.5, 1.2, 1.2, 2.0]),
    "Aggressive Investors": np.array([0.8, 1.0, 2.0, 2.5, 2.5, 2.0, 2.0, 1.5]),
    "Pre-Retirees": np.array([3.5, 2.5, 1.0, 0.8, 0.8, 0.6, 0.6, 2.0]),
    "Second Chance Retirees": np.array([1.0, 1.5, 2.0, 2.0, 1.8, 1.5, 1.5, 1.2])
}

# -------------------------
# 5. Monte Carlo Simulation with Weight Constraints
# -------------------------
np.random.seed(42)
n_portfolios = 100000
best_portfolios = {}

for segment, (low, high) in bands.items():
    alpha = segment_alphas[segment]
    portfolios = []

    for _ in range(n_portfolios):
        weights = np.random.dirichlet(alpha)

        # Enforce minimum Bonds allocation for conservative segments
        if segment in ["Conservative Investors", "Pre-Retirees"]:
            weights[0] = max(weights[0], 0.2)  # Bonds >= 20%
            weights /= weights.sum()

        port_return = np.dot(weights, mean_returns)
        port_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        sharpe_ratio = (port_return - risk_free_rate) / port_volatility

        if low <= port_volatility <= high:
            portfolios.append([port_return, port_volatility, sharpe_ratio, *weights])

    if portfolios:
        df_segment = pd.DataFrame(
            portfolios,
            columns=['Return', 'Volatility', 'Sharpe'] + [f"{a}_wt" for a in assets]
        )

        top5 = df_segment.nlargest(5, 'Sharpe')
        avg_weights = top5[[f"{a}_wt" for a in assets]].mean().values

        port_return = np.dot(avg_weights, mean_returns)
        port_volatility = np.sqrt(np.dot(avg_weights.T, np.dot(cov_matrix, avg_weights)))
        sharpe_ratio = (port_return - risk_free_rate) / port_volatility

        best_portfolios[segment] = {
            'Return': port_return,
            'Volatility': port_volatility,
            'Sharpe': sharpe_ratio,
            'Weights': dict(zip(assets, avg_weights))
        }

# -------------------------
# 6. Display Results
# -------------------------
for segment, p in best_portfolios.items():
    print(f"\n{segment} Portfolio:")
    print(f" Expected Return: {p['Return']:.2%}")
    print(f" Volatility: {p['Volatility']:.2%}")
    print(f" Sharpe Ratio: {p['Sharpe']:.2f}")
    for a, w in p['Weights'].items():
        print(f"  {a} Weight: {w:.2%}")



Conservative Investors Portfolio:
 Expected Return: 7.07%
 Volatility: 9.96%
 Sharpe Ratio: 0.51
  Bonds Weight: 33.09%
  Market ETF Weight: 19.85%
  Large Cap Weight: 6.90%
  Mid Cap Weight: 0.41%
  Small Cap Weight: 0.09%
  Foreign Ex US Weight: 0.40%
  Emerging Weight: 0.06%
  Commodities Weight: 39.20%

Balanced Investors Portfolio:
 Expected Return: 9.50%
 Volatility: 11.90%
 Sharpe Ratio: 0.63
  Bonds Weight: 6.04%
  Market ETF Weight: 16.12%
  Large Cap Weight: 21.08%
  Mid Cap Weight: 2.98%
  Small Cap Weight: 2.34%
  Foreign Ex US Weight: 3.06%
  Emerging Weight: 1.14%
  Commodities Weight: 47.23%

Aggressive Investors Portfolio:
 Expected Return: 12.17%
 Volatility: 15.01%
 Sharpe Ratio: 0.68
  Bonds Weight: 1.46%
  Market ETF Weight: 17.80%
  Large Cap Weight: 50.46%
  Mid Cap Weight: 6.91%
  Small Cap Weight: 4.74%
  Foreign Ex US Weight: 5.47%
  Emerging Weight: 2.70%
  Commodities Weight: 10.47%

Pre-Retirees Portfolio:
 Expected Return: 6.97%
 Volatility: 9.93%
 Sharpe 